# **LangChain으로 체인 구조 이해하기 (Chain)**

LangChain에서 Chain은 단일 작업이 아닌 복합적인 사고 과정을 설계하고 실행할 수 있도록 돕는 핵심 추상화 구조입니다. LLM의 기본 사용은 단일 입력에 대한 단일 응답에 그치는 경우가 많지만, 실제 업무에서는 다양한 입력을 가공하거나, 여러 단계에 걸쳐 논리를 전개하고, 중간 결과를 다음 작업에 넘겨주는 흐름이 필요합니다. 이런 복잡한 과정을 체계화하고 재사용 가능하게 묶은 것이 Chain입니다.

Chain은 단순히 프롬프트와 모델을 연결하는 것을 넘어서, 입력 처리, 프롬프트 구성, LLM 호출, 결과 해석 및 후처리에 이르는 일련의 과정을 하나의 파이프라인으로 추상화합니다. 이로 인해 사용자는 매번 LLM 호출 시 프롬프트를 수작업으로 구성할 필요 없이, 재사용 가능한 구성 요소로서 체인을 호출하기만 하면 됩니다.

## LECL

LangChain의 **LCEL**은 **LangChain Expression Language**의 약자로, LangChain에서 연산자 파이프(`|`)로 연결하여 더욱 직관적이고 간결하게 구성할 수 있도록 도와주는 경량 함수 합성 언어입니다. 

## Chain의 장점

Chain을 사용하면 프롬프트와 모델이 분리되어 재사용 및 유지보수 용이합니다. 또한 체계적인 흐름 추적 가능하고 다른 Chain과도 손쉽게 연결 가능합니다.

지금까지 우리는 다음과 같은 순서로 모델에게 질문했습니다.

**[현재 과정]**
<ol>
<li>ChatOpenAI를 이용한 모델 불러오기 (llm)
</li>
<li>
ChatPromptTemplate를 이용한 프롬프트 템플릿 준비하기 (chat_prompt_template)
</li>
<li><b>chat_prompt_template.format_messages를 이용한 프롬프트 생성 (chat_prompt)</b>
</li>
<li><b>llm.invoke를 이용한 답변 생성 (result)
</b></ol>

1, 2번 과정은 우리가 필요한 모델이나 프롬프트를 최초로 생성해야 하는 과정이라 생략은 어려울 것 같습니다. 하지만 3, 4번 과정은 결합함으로써 단계를 줄일 수 있을 것 같습니다. 

**[결합된 과정]**
1. ChatOpenAI를 이용한 모델 불러오기 (llm)
2. ChatPromptTemplate를 이용한 프롬프트 템플릿 준비하기 (chat_prompt_template)
3. **(new) llm.invoke와 프롬프트를 이용한 답변 생성**

그럼 이제 본격적으로 Chain을 사용한 예시를 살펴보겠습니다.

## Chain 사용 예시

### 모델 호출하기

OpenAI API 사용을 위한 API KEY를 등록합니다. 

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = None

ChatOpenAI를 이용하여 모델을 불러옵니다. 항상 일관된 답변을 얻기 위해 `temperature`를 0으로 설정합니다.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name = "gpt-4o-mini",
    temperature = 0
)

### 프롬프트 템플릿 생성하기

`PromptTemplate`을 활용해 프롬프트 구성해 보겠습니다.

In [ ]:
# 프롬프트 생성하기
from langchain_core.prompts import PromptTemplate

template = "{product}를(을) {task}하기 위한 추천 문구를 작성해 줘"

prompt = PromptTemplate(
    input_variables = ["product", "task"],
    template = template
)

Chain을 사용하지 않는 방법으로는 `format()` 메소드를 이용하여 프롬프트를 생성하고 `invoke()` 메소드를 이용하여 모델에 질문을 했습니다. 

In [ ]:
# 현재 과정
llm.invoke(prompt.format(product="노트북", task="충전"))

In [ ]:
contents = llm.invoke(prompt.format(product="노트북", task="충전"))
contents.content

이제 체인을 이용하여 현재 과정을 조금 더 개선해 보겠습니다. `llm`과 `prompt`를 연결하여 해당 과정을 축소할 수 있습니다. 

모델과 프롬프트를 엮을 때는 `prompt | model` 순으로 배치합니다. 

참고로 prompt와 sample_llm 사이에 있는 기호는 알파벳(l,i,I)이 아니라 엔터 위에 있는 기호입니다. (shift + \\)

In [ ]:
chain = prompt | llm

이제 이 과정을 통해 어떻게 변화되었는지 확인해 보겠습니다. 

In [ ]:
# 개선 과정
chain.invoke({"product":"노트북", "task":"충전" })

이 외에도 LangChain에는 수많은 [체인 종류](https://python.langchain.com/api_reference/langchain/chains.html)가 있습니다. 사용 목적에 따라 다양하게 Chain을 활용해 보세요.